<a href="https://colab.research.google.com/github/lucasalechilet/Curso_FCD/blob/main/L5_Data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

0. PREPARACIÓN DEL ENTORNO

In [1]:
# link a github https://github.com/lucasalechilet/Curso_FCD/blob/main/L5_Data_wrangling.ipynb

import pandas as pd
import numpy as np
import seaborn as sns


df_base = sns.load_dataset('titanic')
df_base.to_csv('titanic_crudo.csv', index=False) #cargamos dataset titanic y generamos un csv



1. Carga y exploración de datos

In [2]:

df = pd.read_csv('titanic_crudo.csv') #leemos el csv que generamos anteriormente

print("exploración inicial")
print(df.head(3))  #vemos las primeras 3 filas
print("\información del dataset \n")
df.info() #vemos la información global del dataset


print("Valores nulos por columna")
print(df.isnull().sum())   # sumamos valores nulos por columna
print(f"\nTotal de filas duplicadas exactas: {df.duplicated().sum()}")

exploración inicial
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
\información del dataset 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null   

<>:5: SyntaxWarning: invalid escape sequence '\i'
<>:5: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_16774/4264690553.py:5: SyntaxWarning: invalid escape sequence '\i'
  print("\información del dataset \n")


2. Limpieza y Transformación de Datos

In [3]:
# Usamos la mediana para imputar la edad, ya que evita el sesgo de valores extremos
mediana_edad = df['age'].median() #se calcula la mediana sobre los valores
df['age'] = df['age'].fillna(mediana_edad) #se llena con la mediana calculada

# la columna 'deck' tiene muchos valores nulos por lo que la eliminaremos
df = df.drop(columns=['deck'])

# para los pocos nulos en 'embarked' (puerto), eliminamos esas filas específicas
df = df.dropna(subset=['embarked', 'embark_town'])

# eliminamos registros duplicados
df = df.drop_duplicates()

# mapeamos el género a valores binarios (0 y 1) para facilitar procesamiento
mapeo_sexo = {'male': 0, 'female': 1}  #0 si male, 1 si female
df['sexo_num'] = df['sex'].map(mapeo_sexo)

3. Optimización y estructuración

In [4]:

columnas_utiles = ['survived', 'pclass', 'sex', 'sexo_num', 'age', 'fare', 'embarked']
df_limpio = df[columnas_utiles].copy()  #seleccionamos las columnas mencionadas arriba, y trabajamos sobre este nuevo dataset

df_limpio = df_limpio.rename(columns={
    'survived': 'sobrevivio',
    'pclass': 'clase_ticket',
    'sex': 'sexo_texto',
    'age': 'edad',
    'fare': 'tarifa_usd',
    'embarked': 'puerto_origen'   #traducimos estas columnas a español
})

df_adultos = df_limpio[df_limpio['edad'] >= 18] #filtrado de datos de pasajeros con edad >18

# agrupación y agregación: ¿Cuál fue la tasa de supervivencia y tarifa promedio por clase?
resumen_clases = df_adultos.groupby('clase_ticket').agg(
    tasa_supervivencia=('sobrevivio', 'mean'), # mean sobre booleanos da el porcentaje
    tarifa_promedio=('tarifa_usd', 'mean'),
    total_pasajeros=('sobrevivio', 'count')
).reset_index()

# convertimos la tasa a porcentaje para mejor lectura
resumen_clases['tasa_supervivencia'] = (resumen_clases['tasa_supervivencia'] * 100).round(2)
resumen_clases['tarifa_promedio'] = resumen_clases['tarifa_promedio'].round(2)

print("\n resumen de supervivencia por clase\n")
print(resumen_clases)


 resumen de supervivencia por clase

   clase_ticket  tasa_supervivencia  tarifa_promedio  total_pasajeros
0             1               61.22            84.36              196
1             2               43.97            21.22              141
2             3               23.31            11.82              326


4.Exportación de datos

In [5]:
# Exportamos dataframe limpio a csv
df_adultos.to_csv('titanic_adultos_limpio.csv', index=False)

# Exportar el resumen agrupado a Excel
resumen_clases.to_excel('reporte_supervivencia_clases.xlsx', index=False, sheet_name='Resumen')


5. Conclusiones

Usar el dataset del Titanic en un procesode Data Wrangling demuestra que los datos crudos rara vez están listos para contar una historia real. Si hubiéramos intentado calcular el promedio de edad o entrenar un modelo predictivo con el archivo original, los valores nulos habrían arrojado errores o nos habrían llevado al sesgo.

Utilizar Pandas nos permite no solo rescatar información valiosa (como imputar edades de forma inteligente), sino también transformar texto en variables codificadas y desechar variables inútiles. En definitiva, este proceso es el filtro de calidad que asegura que los análisis y agrupaciones finales reflejen la realidad del escenario de manera confiable.